# Spark MLlib

MLlib is Spark's built-in machine learning library. It provides tools for the **entire ML workflow**:
feature engineering, model training, evaluation, and pipelines — all distributed across your cluster.

In this module you'll learn:
1. **Feature preparation** — VectorAssembler, StringIndexer, OneHotEncoder
2. **Regression** — predicting a continuous value (salary)
3. **Classification** — predicting a category (region)
4. **Pipelines** — chaining transformers and estimators
5. **Evaluation** — measuring model quality
6. **Hyperparameter tuning** — CrossValidator + ParamGridBuilder

**Key concept**: MLlib works with DataFrames. Every model expects a column of type `Vector` (usually called `features`) and a label column. Transformers and Estimators are composable via Pipelines.

In [18]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, datediff, lit, current_date, year, month

spark = SparkSession.builder \
    .appName("Module 09 - Spark MLlib") \
    .master("local[*]") \
    .getOrCreate()

employees = spark.read.csv("../data/employees.csv", header=True, inferSchema=True)
departments = spark.read.csv("../data/departments.csv", header=True, inferSchema=True)
sales = spark.read.csv("../data/sales.csv", header=True, inferSchema=True)

employees.show(5)
sales.show(5)

+-----------+-------------+-------------+------+----------+-------------+
|employee_id|         name|department_id|salary| hire_date|         city|
+-----------+-------------+-------------+------+----------+-------------+
|          1|   Alice Chen|          101| 92000|2019-03-15|San Francisco|
|          2| Bob Martinez|          102| 78000|2020-07-01|     New York|
|          3|Carol Johnson|          101|105000|2017-11-20|San Francisco|
|          4|    David Kim|          103| 67000|2021-01-10|      Chicago|
|          5|Eve Rodriguez|          104| 72000|2020-09-05|     New York|
+-----------+-------------+-------------+------+----------+-------------+
only showing top 5 rows
+-------+-----------+--------+------+--------+----------+-------+
|sale_id|employee_id| product|amount|quantity| sale_date| region|
+-------+-----------+--------+------+--------+----------+-------+
|      1|          4|Widget A|1250.0|       5|2024-01-05|Central|
|      2|         10|Widget B| 890.5|       3|

---
## Core Abstraction: Transformer vs Estimator

Before diving into specific tools, understand the **two building blocks** of every MLlib workflow:

| | **Transformer** | **Estimator** |
|---|---|---|
| **What it does** | Takes a DataFrame, returns a **new DataFrame** with extra column(s) | Takes a DataFrame, **learns** from it, returns a **Transformer** (a trained model) |
| **Key method** | `.transform(df)` | `.fit(df)` → produces a Transformer |
| **Examples** | `VectorAssembler`, fitted `StringIndexerModel`, fitted `LinearRegressionModel` | `StringIndexer`, `LinearRegression`, `DecisionTreeClassifier`, `Pipeline` |

**The lifecycle pattern:**

```
Estimator  --.fit(training_df)-->  Model (a Transformer)  --.transform(new_df)-->  Predictions
```

For example, `LinearRegression` is an **Estimator**. Calling `.fit(train)` learns the coefficients and returns a `LinearRegressionModel`, which is a **Transformer**. You then call `.transform(test)` on the model to get predictions.

Some stages like `VectorAssembler` are **pure Transformers** — they don't need to learn anything, they just reshape columns. Others like `StringIndexer` are **Estimators** — they must first `.fit()` to discover the set of unique strings, then the resulting `StringIndexerModel` can `.transform()` new data.

A **Pipeline** itself is an Estimator: `.fit()` runs each stage in order, and the result is a `PipelineModel` (a Transformer) that applies all trained stages at once.

---
## Concept 1: Feature Preparation with VectorAssembler

MLlib models expect all input features packed into a **single Vector column**.
`VectorAssembler` combines multiple numeric columns into one.

Think of it like this: if you have columns `[age, height, weight]`, VectorAssembler creates a new column `features` containing `[25.0, 170.0, 68.0]` for each row.

In [19]:
from pyspark.ml.feature import VectorAssembler

# Let's predict salary from department_id and tenure (days since hire)
emp_features = employees.withColumn(
    "tenure_days",
    datediff(lit("2026-12-01"), col("hire_date"))
)

assembler = VectorAssembler(
    inputCols=["department_id", "tenure_days"],
    outputCol="features"
)

assembled = assembler.transform(emp_features)
assembled.select("name", "department_id", "tenure_days", "features", "salary").show(10, truncate=False)

+-------------+-------------+-----------+--------------+------+
|name         |department_id|tenure_days|features      |salary|
+-------------+-------------+-----------+--------------+------+
|Alice Chen   |101          |2818       |[101.0,2818.0]|92000 |
|Bob Martinez |102          |2344       |[102.0,2344.0]|78000 |
|Carol Johnson|101          |3298       |[101.0,3298.0]|105000|
|David Kim    |103          |2151       |[103.0,2151.0]|67000 |
|Eve Rodriguez|104          |2278       |[104.0,2278.0]|72000 |
|Frank Wilson |105          |3145       |[105.0,3145.0]|95000 |
|Grace Lee    |101          |3745       |[101.0,3745.0]|110000|
|Henry Brown  |106          |1723       |[106.0,1723.0]|62000 |
|Irene Davis  |102          |2723       |[102.0,2723.0]|83000 |
|Jack Thompson|103          |2197       |[103.0,2197.0]|71000 |
+-------------+-------------+-----------+--------------+------+
only showing top 10 rows


Notice the `features` column is a **DenseVector**. Each row has `[department_id, tenure_days]` packed together. This is what every MLlib model consumes.

In [20]:
# Your code here
sales_assembler = VectorAssembler(
    inputCols=["quantity", "amount"],
    outputCol="features"
)

sales_assembled = sales_assembler.transform(sales)
sales_assembled.select("product", "quantity", "amount", "features").show(5, truncate=False)

+--------+--------+------+------------+
|product |quantity|amount|features    |
+--------+--------+------+------------+
|Widget A|5       |1250.0|[5.0,1250.0]|
|Widget B|3       |890.5 |[3.0,890.5] |
|Gadget X|1       |2100.0|[1.0,2100.0]|
|Widget A|2       |625.0 |[2.0,625.0] |
|Gadget Y|2       |1780.0|[2.0,1780.0]|
+--------+--------+------+------------+
only showing top 5 rows


In [7]:
# --- Solution ---
sales_assembler = VectorAssembler(
    inputCols=["quantity", "amount"],
    outputCol="features"
)

sales_assembled = sales_assembler.transform(sales)
sales_assembled.select("product", "quantity", "amount", "features").show(5, truncate=False)

+--------+--------+------+------------+
|product |quantity|amount|features    |
+--------+--------+------+------------+
|Widget A|5       |1250.0|[5.0,1250.0]|
|Widget B|3       |890.5 |[3.0,890.5] |
|Gadget X|1       |2100.0|[1.0,2100.0]|
|Widget A|2       |625.0 |[2.0,625.0] |
|Gadget Y|2       |1780.0|[2.0,1780.0]|
+--------+--------+------+------------+
only showing top 5 rows


---
## Concept 2: Encoding Categorical Features

ML models need **numbers**, not strings. For categorical columns like `city` or `product`:

1. **StringIndexer** — converts strings to numeric indices (e.g., `"Chicago"` → `0.0`, `"New York"` → `1.0`)
2. **OneHotEncoder** — converts indices to binary vectors (e.g., `0.0` → `[1,0,0]`, `1.0` → `[0,1,0]`)

Use StringIndexer alone for **tree-based models** (Decision Trees, Random Forest).  
Use StringIndexer + OneHotEncoder for **linear models** (Linear Regression, Logistic Regression).

In [21]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

# Step 1: StringIndexer — city names to numbers
city_indexer = StringIndexer(inputCol="city", outputCol="city_index")
indexed = city_indexer.fit(employees).transform(employees)
indexed.select("name", "city", "city_index").show(10)

# Step 2: OneHotEncoder — index to binary vector
city_encoder = OneHotEncoder(inputCol="city_index", outputCol="city_vec")
encoded = city_encoder.fit(indexed).transform(indexed)
encoded.select("name", "city", "city_index", "city_vec").show(10, truncate=False)

+-------------+-------------+----------+
|         name|         city|city_index|
+-------------+-------------+----------+
|   Alice Chen|San Francisco|       0.0|
| Bob Martinez|     New York|       2.0|
|Carol Johnson|San Francisco|       0.0|
|    David Kim|      Chicago|       1.0|
|Eve Rodriguez|     New York|       2.0|
| Frank Wilson|San Francisco|       0.0|
|    Grace Lee|San Francisco|       0.0|
|  Henry Brown|      Chicago|       1.0|
|  Irene Davis|     New York|       2.0|
|Jack Thompson|      Chicago|       1.0|
+-------------+-------------+----------+
only showing top 10 rows
+-------------+-------------+----------+-------------+
|name         |city         |city_index|city_vec     |
+-------------+-------------+----------+-------------+
|Alice Chen   |San Francisco|0.0       |(2,[0],[1.0])|
|Bob Martinez |New York     |2.0       |(2,[],[])    |
|Carol Johnson|San Francisco|0.0       |(2,[0],[1.0])|
|David Kim    |Chicago      |1.0       |(2,[1],[1.0])|
|Eve Rodriguez|N

In [12]:
# Your code here
product_indexer = StringIndexer(inputCol="product", outputCol="product_index")
product_indexed = product_indexer.fit(sales).transform(sales)

product_encoder = OneHotEncoder(inputCol="product_index", outputCol="product_vec")
product_encoded = product_encoder.fit(product_indexed).transform(product_indexed)

product_encoded.select("product", "product_index", "product_vec").show(10, truncate=False)

+--------+-------------+-------------+
|product |product_index|product_vec  |
+--------+-------------+-------------+
|Widget A|0.0          |(5,[0],[1.0])|
|Widget B|1.0          |(5,[1],[1.0])|
|Gadget X|3.0          |(5,[3],[1.0])|
|Widget A|0.0          |(5,[0],[1.0])|
|Gadget Y|4.0          |(5,[4],[1.0])|
|Widget C|2.0          |(5,[2],[1.0])|
|Gadget X|3.0          |(5,[3],[1.0])|
|Widget B|1.0          |(5,[1],[1.0])|
|Widget A|0.0          |(5,[0],[1.0])|
|Gadget Z|5.0          |(5,[],[])    |
+--------+-------------+-------------+
only showing top 10 rows


In [11]:
# --- Solution ---
product_indexer = StringIndexer(inputCol="product", outputCol="product_index")
product_indexed = product_indexer.fit(sales).transform(sales)

product_encoder = OneHotEncoder(inputCol="product_index", outputCol="product_vec")
product_encoded = product_encoder.fit(product_indexed).transform(product_indexed)

product_encoded.select("product", "product_index", "product_vec").show(10, truncate=False)

+--------+-------------+-------------+
|product |product_index|product_vec  |
+--------+-------------+-------------+
|Widget A|0.0          |(5,[0],[1.0])|
|Widget B|1.0          |(5,[1],[1.0])|
|Gadget X|3.0          |(5,[3],[1.0])|
|Widget A|0.0          |(5,[0],[1.0])|
|Gadget Y|4.0          |(5,[4],[1.0])|
|Widget C|2.0          |(5,[2],[1.0])|
|Gadget X|3.0          |(5,[3],[1.0])|
|Widget B|1.0          |(5,[1],[1.0])|
|Widget A|0.0          |(5,[0],[1.0])|
|Gadget Z|5.0          |(5,[],[])    |
+--------+-------------+-------------+
only showing top 10 rows


---
## Concept 3: Linear Regression — Predicting Salary

Let's build a model to predict employee salary from tenure and city.

The ML workflow:
1. **Prepare features** (encode categoricals, assemble into vector)
2. **Split data** into train/test sets
3. **Fit** the model on training data
4. **Predict** on test data
5. **Evaluate** the results

In [23]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Prepare the data
emp_ml = employees.withColumn(
    "tenure_days",
    datediff(lit("2024-12-01"), col("hire_date"))
)

# Encode city
city_indexer = StringIndexer(inputCol="city", outputCol="city_index")
emp_ml = city_indexer.fit(emp_ml).transform(emp_ml)

city_encoder = OneHotEncoder(inputCol="city_index", outputCol="city_vec")
emp_ml = city_encoder.fit(emp_ml).transform(emp_ml)

# Assemble all features
assembler = VectorAssembler(
    inputCols=["tenure_days", "city_vec"],
    outputCol="features"
)
emp_ml = assembler.transform(emp_ml)

# Rename salary as label (MLlib convention)
emp_ml = emp_ml.withColumnRenamed("salary", "label")

# Train/test split (80/20)
train, test = emp_ml.randomSplit([0.8, 0.2], seed=42)
print(f"Training rows: {train.count()}, Test rows: {test.count()}")

Training rows: 23, Test rows: 7


In [24]:
# Train the model
lr = LinearRegression(maxIter=10, regParam=0.01)
lr_model = lr.fit(train)

# Print model coefficients
print(f"Coefficients: {lr_model.coefficients}")
print(f"Intercept:    {lr_model.intercept:.2f}")

Coefficients: [15.395124777740932,7423.253227972979,-3257.716945197043]
Intercept:    52530.94


In [25]:
# Predict on test data
predictions = lr_model.transform(test)
predictions.select("name", "label", "prediction").show(truncate=False)

# Evaluate
evaluator = RegressionEvaluator(metricName="rmse")
rmse = evaluator.evaluate(predictions)

evaluator_r2 = RegressionEvaluator(metricName="r2")
r2 = evaluator_r2.evaluate(predictions)

print(f"\nRMSE: {rmse:.2f}  (average prediction error in dollars)")
print(f"R²:   {r2:.4f}  (1.0 = perfect, 0.0 = no better than guessing the mean)")

+-------------+------+------------------+
|name         |label |prediction        |
+-------------+------+------------------+
|Carol Johnson|105000|99488.8753210095  |
|Grace Lee    |110000|106370.49609665968|
|Irene Davis  |83000 |83213.42534583548 |
|Noah Harris  |64000 |62097.363658458955|
|Tina Young   |68000 |68649.63730609257 |
|Xavier Adams |74000 |74290.30248242978 |
|Diana Flores |70000 |63698.45663534402 |
+-------------+------+------------------+


RMSE: 3534.06  (average prediction error in dollars)
R²:   0.9572  (1.0 = perfect, 0.0 = no better than guessing the mean)


**Reading the results**: RMSE tells you the average prediction error in the same units as the label (dollars). R² tells you what fraction of the variance your model explains. With only 30 rows and 2 features, don't expect great numbers — the point is understanding the workflow.

In [17]:
# Your code here
sales_ml = VectorAssembler(
    inputCols=["quantity"],
    outputCol="features"
).transform(sales).withColumnRenamed("amount", "label")

train_s, test_s = sales_ml.randomSplit([0.8, 0.2], seed=42)
lr_sales = LinearRegression(maxIter=10, regParam=0.01)
lr_sales_model = lr_sales.fit(train_s)
preds_s = lr_sales_model.transform(test_s)
preds_s.select("product", "quantity", "label", "prediction").show(10)

rmse_s = RegressionEvaluator(metricName="rmse").evaluate(preds_s)
print(f"RMSE: {rmse_s:.2f}")

+--------+--------+-------+------------------+
| product|quantity|  label|        prediction|
+--------+--------+-------+------------------+
|Gadget X|       1| 2100.0| 2164.736901523851|
|Gadget X|       1| 2100.0| 2164.736901523851|
|Widget A|       7| 1875.0|1936.0910808618619|
|Widget B|       5|1484.17|2012.3063544158583|
|Gadget Y|       2| 1780.0| 2126.629264746853|
|Widget C|       2|  150.0| 2126.629264746853|
|Gadget X|       3| 6300.0|2088.5216279698548|
|Widget B|       2| 593.67| 2126.629264746853|
|Widget B|       6|1780.83|  1974.19871763886|
|Gadget Z|       1| 3200.0| 2164.736901523851|
+--------+--------+-------+------------------+
only showing top 10 rows
RMSE: 1580.84


In [13]:
# --- Solution ---
sales_ml = VectorAssembler(
    inputCols=["quantity"],
    outputCol="features"
).transform(sales).withColumnRenamed("amount", "label")

train_s, test_s = sales_ml.randomSplit([0.8, 0.2], seed=42)

lr_sales = LinearRegression(maxIter=10, regParam=0.01)
lr_sales_model = lr_sales.fit(train_s)

preds_s = lr_sales_model.transform(test_s)
preds_s.select("product", "quantity", "label", "prediction").show(10)

rmse_s = RegressionEvaluator(metricName="rmse").evaluate(preds_s)
print(f"RMSE: {rmse_s:.2f}")

+--------+--------+-------+------------------+
| product|quantity|  label|        prediction|
+--------+--------+-------+------------------+
|Gadget X|       1| 2100.0| 2164.736901523851|
|Gadget X|       1| 2100.0| 2164.736901523851|
|Widget A|       7| 1875.0|1936.0910808618619|
|Widget B|       5|1484.17|2012.3063544158583|
|Gadget Y|       2| 1780.0| 2126.629264746853|
|Widget C|       2|  150.0| 2126.629264746853|
|Gadget X|       3| 6300.0|2088.5216279698548|
|Widget B|       2| 593.67| 2126.629264746853|
|Widget B|       6|1780.83|  1974.19871763886|
|Gadget Z|       1| 3200.0| 2164.736901523851|
+--------+--------+-------+------------------+
only showing top 10 rows
RMSE: 1580.84


---
## Concept 4: Classification — Predicting Region

Classification predicts a **category** instead of a number. We'll use a **Decision Tree** to predict which `region` a sale belongs to based on its product, quantity, and amount.

Decision Trees are great for learning because they're interpretable — you can see the split rules.

In [26]:
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Encode the product (categorical) and region (label)
product_indexer = StringIndexer(inputCol="product", outputCol="product_index")
region_indexer = StringIndexer(inputCol="region", outputCol="label")

sales_clf = product_indexer.fit(sales).transform(sales)
sales_clf = region_indexer.fit(sales_clf).transform(sales_clf)

# Assemble features (product_index is fine for tree models — no need to one-hot encode)
assembler_clf = VectorAssembler(
    inputCols=["product_index", "quantity", "amount"],
    outputCol="features"
)
sales_clf = assembler_clf.transform(sales_clf)

# Split
train_c, test_c = sales_clf.randomSplit([0.8, 0.2], seed=42)

# Train decision tree
dt = DecisionTreeClassifier(maxDepth=5, seed=42)
dt_model = dt.fit(train_c)

# Predict
preds_c = dt_model.transform(test_c)
preds_c.select("product", "quantity", "amount", "region", "label", "prediction").show(10)

+--------+--------+-------+-------+-----+----------+
| product|quantity| amount| region|label|prediction|
+--------+--------+-------+-------+-----+----------+
|Gadget X|       1| 2100.0|Central|  2.0|       2.0|
|Gadget X|       1| 2100.0|Central|  2.0|       2.0|
|Widget A|       7| 1875.0|Central|  2.0|       0.0|
|Widget B|       5|1484.17|   East|  1.0|       2.0|
|Gadget Y|       2| 1780.0|   East|  1.0|       2.0|
|Widget C|       2|  150.0|   West|  0.0|       2.0|
|Gadget X|       3| 6300.0|   West|  0.0|       0.0|
|Widget B|       2| 593.67|Central|  2.0|       2.0|
|Widget B|       6|1780.83|   East|  1.0|       0.0|
|Gadget Z|       1| 3200.0|   West|  0.0|       2.0|
+--------+--------+-------+-------+-----+----------+
only showing top 10 rows


In [18]:
# Evaluate accuracy
acc_evaluator = MulticlassClassificationEvaluator(metricName="accuracy")
accuracy = acc_evaluator.evaluate(preds_c)

f1_evaluator = MulticlassClassificationEvaluator(metricName="f1")
f1 = f1_evaluator.evaluate(preds_c)

print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")

# Feature importance — which features matter most?
print(f"\nFeature importances: {dt_model.featureImportances}")
print("  [0] product_index")
print("  [1] quantity")
print("  [2] amount")

# View the tree structure
print(f"\nTree depth: {dt_model.depth}")
print(f"Number of nodes: {dt_model.numNodes}")

Accuracy: 0.2941
F1 Score: 0.2456

Feature importances: (3,[0,1,2],[0.31233621817058943,0.26084125612893255,0.426822525700478])
  [0] product_index
  [1] quantity
  [2] amount

Tree depth: 5
Number of nodes: 17


In [ ]:
# Your code here


In [ ]:
# --- Solution ---
emp_clf = employees.withColumn(
    "tenure_days", datediff(lit("2024-12-01"), col("hire_date"))
)

city_label_indexer = StringIndexer(inputCol="city", outputCol="label")
emp_clf = city_label_indexer.fit(emp_clf).transform(emp_clf)

emp_assembler = VectorAssembler(
    inputCols=["department_id", "salary", "tenure_days"],
    outputCol="features"
)
emp_clf = emp_assembler.transform(emp_clf)

train_e, test_e = emp_clf.randomSplit([0.8, 0.2], seed=42)

dt_emp = DecisionTreeClassifier(maxDepth=5, seed=42)
dt_emp_model = dt_emp.fit(train_e)

preds_e = dt_emp_model.transform(test_e)
preds_e.select("name", "city", "label", "prediction").show()

accuracy_e = MulticlassClassificationEvaluator(metricName="accuracy").evaluate(preds_e)
print(f"Accuracy: {accuracy_e:.4f}")

---
## Concept 5: ML Pipelines

Writing all those steps manually is error-prone. A **Pipeline** chains multiple stages (transformers + estimators) into a single object that can be fit and applied at once.

Benefits:
- **Reproducibility** — same preprocessing for train and test
- **Cleaner code** — one `.fit()` call does everything
- **Serialization** — save/load the entire pipeline

In [ ]:
from pyspark.ml import Pipeline

# Rebuild the salary prediction as a Pipeline
emp_pipe = employees.withColumn(
    "tenure_days", datediff(lit("2024-12-01"), col("hire_date"))
).withColumnRenamed("salary", "label")

# Define pipeline stages
stage_indexer = StringIndexer(inputCol="city", outputCol="city_index")
stage_encoder = OneHotEncoder(inputCol="city_index", outputCol="city_vec")
stage_assembler = VectorAssembler(
    inputCols=["tenure_days", "city_vec"],
    outputCol="features"
)
stage_lr = LinearRegression(maxIter=10, regParam=0.01)

# Build pipeline
pipeline = Pipeline(stages=[
    stage_indexer,
    stage_encoder,
    stage_assembler,
    stage_lr
])

# One call to fit everything
train_p, test_p = emp_pipe.randomSplit([0.8, 0.2], seed=42)
pipeline_model = pipeline.fit(train_p)

# One call to predict
preds_p = pipeline_model.transform(test_p)
preds_p.select("name", "label", "prediction").show()

rmse_p = RegressionEvaluator(metricName="rmse").evaluate(preds_p)
print(f"Pipeline RMSE: {rmse_p:.2f}")

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
clf_pipeline = Pipeline(stages=[
    StringIndexer(inputCol="product", outputCol="product_index"),
    StringIndexer(inputCol="region", outputCol="label"),
    VectorAssembler(inputCols=["product_index", "quantity", "amount"], outputCol="features"),
    DecisionTreeClassifier(maxDepth=5, seed=42)
])

train_cp, test_cp = sales.randomSplit([0.8, 0.2], seed=42)
clf_pipeline_model = clf_pipeline.fit(train_cp)

preds_cp = clf_pipeline_model.transform(test_cp)
preds_cp.select("product", "region", "label", "prediction").show(10)

accuracy_cp = MulticlassClassificationEvaluator(metricName="accuracy").evaluate(preds_cp)
print(f"Pipeline Accuracy: {accuracy_cp:.4f}")

---
## Concept 6: Hyperparameter Tuning with CrossValidator

How do you pick the best `maxDepth` or `regParam`? **Cross-validation** trains the model with different parameter combinations on multiple data folds and picks the best one.

- `ParamGridBuilder` — defines the parameter search space
- `CrossValidator` — runs k-fold CV over the grid

This is where MLlib's distributed computing really shines — each fold and parameter combo can run in parallel.

In [ ]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# Build a pipeline to tune
dt_tunable = DecisionTreeClassifier(seed=42)

tune_pipeline = Pipeline(stages=[
    StringIndexer(inputCol="product", outputCol="product_index"),
    StringIndexer(inputCol="region", outputCol="label"),
    VectorAssembler(inputCols=["product_index", "quantity", "amount"], outputCol="features"),
    dt_tunable
])

# Define the parameter grid
param_grid = ParamGridBuilder() \
    .addGrid(dt_tunable.maxDepth, [3, 5, 7, 10]) \
    .addGrid(dt_tunable.minInstancesPerNode, [1, 2, 5]) \
    .build()

print(f"Total parameter combinations: {len(param_grid)}")

# CrossValidator: 3-fold CV
cv = CrossValidator(
    estimator=tune_pipeline,
    estimatorParamMaps=param_grid,
    evaluator=MulticlassClassificationEvaluator(metricName="accuracy"),
    numFolds=3,
    seed=42
)

# Fit — this trains 12 combos × 3 folds = 36 models!
cv_model = cv.fit(sales)

# Best model results
best_dt = cv_model.bestModel.stages[-1]  # last stage is the DecisionTree
print(f"\nBest maxDepth:           {best_dt.depth}")
print(f"Best model tree nodes:   {best_dt.numNodes}")
print(f"Best cross-val accuracy: {max(cv_model.avgMetrics):.4f}")

In [ ]:
# Your code here


In [ ]:
# --- Solution ---
lr_tunable = LinearRegression(maxIter=10)

tune_lr_pipeline = Pipeline(stages=[
    StringIndexer(inputCol="city", outputCol="city_index"),
    OneHotEncoder(inputCol="city_index", outputCol="city_vec"),
    VectorAssembler(inputCols=["tenure_days", "city_vec"], outputCol="features"),
    lr_tunable
])

lr_param_grid = ParamGridBuilder() \
    .addGrid(lr_tunable.regParam, [0.001, 0.01, 0.1, 1.0]) \
    .build()

lr_cv = CrossValidator(
    estimator=tune_lr_pipeline,
    estimatorParamMaps=lr_param_grid,
    evaluator=RegressionEvaluator(metricName="rmse"),
    numFolds=3,
    seed=42
)

lr_cv_model = lr_cv.fit(emp_pipe)

# Note: for RMSE, lower is better, so best = min
print(f"Cross-val RMSE scores: {[f'{m:.2f}' for m in lr_cv_model.avgMetrics]}")
print(f"Best RMSE: {min(lr_cv_model.avgMetrics):.2f}")

---
## Concept 7: Saving and Loading Models

Trained pipelines can be saved to disk and loaded later — useful for deploying models to production or sharing with teammates.

In [ ]:
# Save the pipeline model
pipeline_model.write().overwrite().save("../output/salary_pipeline_model")

# Load it back
from pyspark.ml import PipelineModel
loaded_model = PipelineModel.load("../output/salary_pipeline_model")

# Use loaded model to predict
loaded_preds = loaded_model.transform(test_p)
loaded_preds.select("name", "label", "prediction").show(5)

print("Model saved and loaded successfully!")

In [ ]:
# Your capstone code here


In [ ]:
# --- Capstone Solution ---

# Re-read fresh copies
employees_fresh = spark.read.csv("../data/employees.csv", header=True, inferSchema=True)
sales_fresh = spark.read.csv("../data/sales.csv", header=True, inferSchema=True)

# Join and engineer features
combined = sales_fresh.join(
    employees_fresh.select("employee_id", "salary", "city", "hire_date"),
    on="employee_id"
).withColumn(
    "tenure_days", datediff(lit("2024-12-01"), col("hire_date"))
).withColumnRenamed("amount", "label")

# Pipeline stages
cap_product_indexer = StringIndexer(inputCol="product", outputCol="product_idx")
cap_product_encoder = OneHotEncoder(inputCol="product_idx", outputCol="product_vec")

cap_region_indexer = StringIndexer(inputCol="region", outputCol="region_idx")
cap_region_encoder = OneHotEncoder(inputCol="region_idx", outputCol="region_vec")

cap_city_indexer = StringIndexer(inputCol="city", outputCol="city_idx")
cap_city_encoder = OneHotEncoder(inputCol="city_idx", outputCol="city_vec")

cap_assembler = VectorAssembler(
    inputCols=["quantity", "salary", "tenure_days", "product_vec", "region_vec", "city_vec"],
    outputCol="features"
)

cap_lr = LinearRegression(maxIter=20, regParam=0.01)

capstone_pipeline = Pipeline(stages=[
    cap_product_indexer, cap_product_encoder,
    cap_region_indexer, cap_region_encoder,
    cap_city_indexer, cap_city_encoder,
    cap_assembler,
    cap_lr
])

# Train and evaluate
train_cap, test_cap = combined.randomSplit([0.8, 0.2], seed=42)
capstone_model = capstone_pipeline.fit(train_cap)

cap_preds = capstone_model.transform(test_cap)
cap_preds.select("product", "region", "quantity", "label", "prediction").show(10)

cap_rmse = RegressionEvaluator(metricName="rmse").evaluate(cap_preds)
cap_r2 = RegressionEvaluator(metricName="r2").evaluate(cap_preds)

print(f"\nCapstone Results:")
print(f"  RMSE: {cap_rmse:.2f}")
print(f"  R²:   {cap_r2:.4f}")

In [ ]:
spark.stop()

---
## What You Learned

- **VectorAssembler** packs multiple columns into one feature vector
- **StringIndexer** + **OneHotEncoder** convert categorical strings to numbers
- **LinearRegression** predicts continuous values; evaluated with RMSE and R²
- **DecisionTreeClassifier** predicts categories; evaluated with accuracy and F1
- **Pipelines** chain transformers and estimators for clean, reproducible workflows
- **CrossValidator** + **ParamGridBuilder** automate hyperparameter tuning
- Models can be **saved and loaded** for reuse

### MLlib Quick Reference

| Task | Class | Key Params |
|------|-------|------------|
| Feature assembly | `VectorAssembler` | `inputCols`, `outputCol` |
| String → index | `StringIndexer` | `inputCol`, `outputCol` |
| Index → one-hot | `OneHotEncoder` | `inputCol`, `outputCol` |
| Regression | `LinearRegression` | `regParam`, `maxIter` |
| Classification | `DecisionTreeClassifier` | `maxDepth`, `minInstancesPerNode` |
| Classification | `RandomForestClassifier` | `numTrees`, `maxDepth` |
| Clustering | `KMeans` | `k`, `seed` |
| Pipeline | `Pipeline` | `stages` |
| Tuning | `CrossValidator` | `numFolds`, `estimatorParamMaps` |